<h1 style="color:#c8a2c8; font-weight:bold; margin-bottom: 0px; padding-bottom: 0px;">ImmoEliza Project</h1>  
<h2 style="color:#c8a2c8; margin-top: 0px; padding-top: 0px;">ML content</h2>

<i><span style="color:orange">Note for the 03/07:</span> try to make all checking prints not blind.</i>

### <span style="color:#c8a2c8; font-weight:bold">2 - Preprocessing</span>

In [1]:
# import libraries

# Standard 
import warnings 
import os
import joblib

# Data manipulation & visualization
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, MinMaxScaler
from sklearn.pipeline import Pipeline
# other sklearn imports
from sklearn.base import BaseEstimator, TransformerMixin

# Display settings
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.3f}".format)
sns.set_theme(style="whitegrid")

# Reproducibility - alwas se a random seed
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

MODEL_DIR_PATH = "../models"

print("✅ Setup complete.")

✅ Setup complete.


In [2]:
# dataset loading

X_TRAIN_RAW_CSV_PATH = "../data/raw/train_test/X_train_raw.csv"
X_TEST_RAW_CSV_PATH = "../data/raw/train_test/X_test_raw.csv"

properties_train = pd.read_csv(X_TRAIN_RAW_CSV_PATH)
properties_test = pd.read_csv(X_TEST_RAW_CSV_PATH)

print("✅ Dataset for train 'properties_train' and Dataset for test 'properties_test' successfully loaded.")

✅ Dataset for train 'properties_train' and Dataset for test 'properties_test' successfully loaded.


<span style="color:#c8a2c8">Test Set</span> has to remain a black box untill testing, but modified according to Train Set.

<h3 style="color:#c8a2c8; margin-bottom: 0px; padding-bottom: 0px;">2.1 - EDA</h3>  
<h4 style="color:#c8a2c8; margin-top: 0px; padding-top: 0px;">properties_train dataset (X_train_raw)</h4>

In [3]:
print(f'Dataset Shape: {properties_train.shape[0]} rows x {properties_train.shape[1]} columns\n')
properties_train.head()


Dataset Shape: 9919 rows x 32 columns



,property_type,property_subtype,living_area_m2,bedrooms,bathrooms,address,postal_code,city,latitude,longitude,building_year,state_of_the_building,furnished,has_garage,parking_count,kitchen_equipped,has_elevator,facades,floors_total,has_garden,garden_area_m2,has_terrace,total_area_m2,epc_score,region,province,nearby_city,km_from_nearby_city,is_nearby_city_prestigious,floor_number,property_url,coord_swapped
0,Apartment,apartment,107.000,3.000,NaN,NaN,5580,Rochefort,50.160,5.222,NaN,NaN,0,1,NaN,NaN,0,NaN,NaN,0,NaN,1,NaN,NaN,Wallonia,Namur,Namur,42.400,0.000,NaN,https://immovlan.be/en/detail/apartment/for-sa...,False
1,House,residence,153.000,4.000,1.000,Van Benedenlaan 33,2800,Mechelen,51.022,4.475,1899.000,NaN,0,0,NaN,Fully equipped,0,2.000,NaN,0,NaN,1,78.000,D,Flanders,Antwerp,Malines,0.500,0.000,NaN,https://immovlan.be/en/detail/residence/for-sa...,False
2,House,residence,164.000,3.000,1.000,NaN,6220,Lambusart,50.455,4.553,NaN,NaN,0,1,1.000,NaN,0,NaN,3.000,1,NaN,1,730.000,F,Wallonia,Hainaut,Charleroi,9.100,0.000,NaN,https://immovlan.be/en/detail/residence/for-sa...,False
3,House,residence,256.000,3.000,1.000,Dennenstraat 61,3945,Ham,51.109,5.192,2005.000,Normal,0,1,NaN,NaN,0,4.000,NaN,1,NaN,0,1655.000,B,Flanders,Limburg,Geel,15.300,0.000,NaN,https://immovlan.be/en/detail/residence/for-sa...,False
4,Apartment,apartment,100.000,2.000,NaN,Rue Henri Caron 42,1070,Anderlecht,50.841,4.301,1962.000,Normal,0,0,NaN,NaN,0,2.000,NaN,0,NaN,0,NaN,C,Brussels,Brussels Capital Region,Bruxelles,3.700,0.000,2.000,https://immovlan.be/en/detail/apartment/for-sa...,False


In [4]:
properties_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 9919 entries, 0 to 9918
Data columns (total 32 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   property_type               9919 non-null   str    
 1   property_subtype            9919 non-null   str    
 2   living_area_m2              9237 non-null   float64
 3   bedrooms                    9597 non-null   float64
 4   bathrooms                   8843 non-null   float64
 5   address                     7950 non-null   str    
 6   postal_code                 9919 non-null   int64  
 7   city                        9919 non-null   str    
 8   latitude                    9917 non-null   float64
 9   longitude                   9917 non-null   float64
 10  building_year               5964 non-null   float64
 11  state_of_the_building       7660 non-null   str    
 12  furnished                   9919 non-null   int64  
 13  has_garage                  9919 non-null   

In [5]:
properties_train.describe(include="all")


,property_type,property_subtype,living_area_m2,bedrooms,bathrooms,address,postal_code,city,latitude,longitude,building_year,state_of_the_building,furnished,has_garage,parking_count,kitchen_equipped,has_elevator,facades,floors_total,has_garden,garden_area_m2,has_terrace,total_area_m2,epc_score,region,province,nearby_city,km_from_nearby_city,is_nearby_city_prestigious,floor_number,property_url,coord_swapped
count,9919,9919,9237.000,9597.000,8843.000,7950,9919.000,9919,9917.000,9917.000,5964.000,7660,9919.000,9919.000,3353.000,2716,9919.000,7527.000,4460.000,9919.000,2201.000,9919.000,5525.000,8361,9919,9919,9917,9917.000,9917.000,2790.000,9919,9919
unique,2,15,NaN,NaN,NaN,7539,NaN,1491,NaN,NaN,NaN,8,NaN,NaN,NaN,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11,3,11,43,NaN,NaN,NaN,9915,2
top,House,residence,NaN,NaN,NaN,Dessauerplein 1,NaN,Liege,NaN,NaN,NaN,Normal,NaN,NaN,NaN,Fully equipped,NaN,NaN,NaN,NaN,NaN,NaN,NaN,C,Wallonia,Liège,Bruxelles,NaN,NaN,NaN,https://immovlan.be/en/detail/residence/for-sa...,False
freq,6442,5218,NaN,NaN,NaN,26,NaN,213,NaN,NaN,NaN,2921,NaN,NaN,NaN,1290,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1669,4745,1361,1123,NaN,NaN,NaN,2,9884
mean,NaN,NaN,172.341,3.075,1.412,NaN,5014.252,NaN,50.722,4.546,1969.568,NaN,0.036,0.450,1.940,NaN,0.190,2.957,3.320,0.537,1179.508,0.657,1481.838,NaN,NaN,NaN,NaN,35.001,0.064,3.408,NaN,NaN
std,NaN,NaN,148.001,2.306,1.227,NaN,2682.302,NaN,0.372,0.819,47.477,NaN,0.187,0.498,32.746,NaN,0.392,0.882,2.973,0.499,5064.902,0.475,4496.044,NaN,NaN,NaN,NaN,386.809,0.244,26.820,NaN,NaN
min,NaN,NaN,12.000,1.000,1.000,NaN,1000.000,NaN,49.507,2.509,1500.000,NaN,0.000,0.000,1.000,NaN,0.000,1.000,1.000,0.000,1.000,0.000,2.000,NaN,NaN,NaN,NaN,0.000,0.000,0.000,NaN,NaN
25%,NaN,NaN,90.000,2.000,1.000,NaN,2570.000,NaN,50.474,4.154,1950.000,NaN,0.000,0.000,1.000,NaN,0.000,2.000,2.000,0.000,117.000,0.000,270.000,NaN,NaN,NaN,NaN,3.200,0.000,1.000,NaN,NaN
50%,NaN,NaN,145.000,3.000,1.000,NaN,4960.000,NaN,50.777,4.458,1974.000,NaN,0.000,0.000,1.000,NaN,0.000,3.000,3.000,1.000,350.000,1.000,630.000,NaN,NaN,NaN,NaN,8.400,0.000,2.000,NaN,NaN
75%,NaN,NaN,205.000,4.000,2.000,NaN,7100.000,NaN,51.010,5.231,2007.000,NaN,0.000,1.000,1.000,NaN,0.000,4.000,4.000,1.000,900.000,1.000,1299.000,NaN,NaN,NaN,NaN,16.900,0.000,3.000,NaN,NaN


In [6]:
columns = [
    'property_type', 
    'property_subtype', 
    'state_of_the_building', 
    'furnished', 
    'has_garage', 
    'kitchen_equipped', 
    'has_elevator', 
    'has_garden', 
    'has_terrace', 
    'epc_score', 
    'region', 
    'province', 
    'is_nearby_city_prestigious', 
    'nearby_city'
]
for column in columns:
    elements = "\n".join(properties_train[column].dropna().unique().astype(str))
    print(f'{column} in the dataset contains {properties_train[column].nunique()} unique values:\n{elements}')
    print(f'\nDatatype of vales in the column is {properties_train[column].dropna().unique().dtype}.')

property_type in the dataset contains 2 unique values:
Apartment
House

Datatype of vales in the column is str.
property_subtype in the dataset contains 15 unique values:
apartment
residence
cottage
villa
chalet
mixed-building
penthouse
duplex
bungalow
loft
ground-floor
studio
triplex
mansion
master-house

Datatype of vales in the column is str.
state_of_the_building in the dataset contains 8 unique values:
Normal
To renovate
New
Excellent
Fully renovated
To restore
To demolish
Under construction

Datatype of vales in the column is str.
furnished in the dataset contains 2 unique values:
0
1

Datatype of vales in the column is int64.
has_garage in the dataset contains 2 unique values:
1
0

Datatype of vales in the column is int64.
kitchen_equipped in the dataset contains 4 unique values:
Fully equipped
Partially equipped
Super equipped
Not equipped

Datatype of vales in the column is str.
has_elevator in the dataset contains 2 unique values:
0
1

Datatype of vales in the column is int64

<i><span style="color:orange">Note of Wednesday 01/07:</span> <code>'city'</code> and <code>'nearby_city'</code> has too many str values for one-hot encoding, for model robustness sake I'm dropping it in stage 2.2 below for the moment, but it could be turned to a continuous numerical column substituting each city with the mean price value of properties in that specific city.</i>

In [7]:
properties_train['is_nearby_city_prestigious'] = properties_train['is_nearby_city_prestigious'].astype('Int64')
properties_test['is_nearby_city_prestigious'] = properties_test['is_nearby_city_prestigious'].astype('Int64')

print("✅ Values in 'is_nearby_city_prestigious' columns successfully converted to Int64 in both datasets.")

✅ Values in 'is_nearby_city_prestigious' columns successfully converted to Int64 in both datasets.


<i><span style="color:green">DONE:</span> make <code>'is_nearby_city_prestigious'</code> dtype Int64 rather than float64.</i>

In [ ]:
# properties_train[column] = properties_train[column].astype('Int64')

<h3 style="color:#c8a2c8; margin-bottom: 0px; padding-bottom: 0px;">2.2 - Drop Useless Columns</h3>  
<h4 style="color:#c8a2c8; margin-top: 0px; padding-top: 0px;">both properties_train (X_train_raw) and properties_test (X_test_raw)</h4>  

I'm going to remove the columns:  
- `'property_url'`;  
- `'address'`;  
- `'nearby_city'`;  
- `'price_per_m2'` &rarr; already removed in Data Splitting notebook.

In [8]:
# removal of columns that a model can't calculate directly:
# 'property_url', 'address'

# removal of 'price_per_m2' also to prevent Data Leakage

# create a dedicated config dictionary
COLS_TO_DROP = {
    "property_url": "Unique for each property - no predictive power",
    "address": "Unique for each property - no predictive power",
    "city": "Too many values - would affect negatively predictive power",
    "nearby_city": "Too many values - would affect negatively predictive power"
}

'''
properties_train
----------------
'''
# Drop only columns that exist (safe for re-runs)
cols_present = [c for c in COLS_TO_DROP if c in properties_train.columns]
properties_train = properties_train.drop(columns=cols_present)

print(f'Dropped {len(cols_present)} column(s) from properties_train dataset: {", ".join(cols_present)}')
print(f'Remaining dataset Shape: {properties_train.shape}')

'''
properties_test
----------------
'''
# Drop only columns that exist (safe for re-runs)
cols_present = [c for c in COLS_TO_DROP if c in properties_test.columns]
properties_test = properties_test.drop(columns=cols_present)

print(f'Dropped {len(cols_present)} column(s) from properties_test dataset: {", ".join(cols_present)}')
print(f'Remaining dataset Shape: {properties_test.shape}')

Dropped 4 column(s) from properties_train dataset: property_url, address, city, nearby_city
Remaining dataset Shape: (9919, 28)
Dropped 4 column(s) from properties_test dataset: property_url, address, city, nearby_city
Remaining dataset Shape: (2480, 28)


<h3 style="color:#c8a2c8; margin-bottom: 0px; padding-bottom: 0px;">2.3 - Feature Engineering — Creating New Features</h3>  
<h4 style="color:#c8a2c8; margin-top: 0px; padding-top: 0px;">both properties_train (X_train_raw) and properties_test (X_test_raw)</h4>  

I want to add:
-  a categorical column named `'urban_density'` reporting the level of urban density (between High, Medium, Low) according to corresponding `'postal_code'` of the property - I start from having it numerical, by just calculating density;  
- a column named `'property_age'` (2026 - df['building_year']);  
- a column named `'living_area_ratio'` (df['living_area_m2'] / df['total_area_m2']);  
- a column named `'bedroom_density'` (df['bedrooms'] / df['living_area_m2']).

<i><span style="color:orange">Note for the 03/07:</span> substitute 2026 with Pandas dates handling for inserting the current year.</i>

In [9]:
'''
===========================
'urban_density' column
===========================
'''

# importing different pandas dataframe with demografical data of Belgium
# source: StatBel

# https://statbel.fgov.be/en/open-data/population-statistical-sector-12
DEMOG_BE_DATASET = "../data/raw/StatBel/population_data/OPENDATA_SECTOREN_2025_NEW.txt"
# https://statbel.fgov.be/en/open-data/address-file-statistical-sector
NIS_CODES_DATASET = "../data/raw/StatBel/population_data/TF_RGN_ADDRESSES_STAT_SECTORS_20240401.txt"

demog_df = pd.read_csv(DEMOG_BE_DATASET, sep='|', encoding='cp1252')
nis_df = pd.read_csv(NIS_CODES_DATASET, sep='|', encoding='cp1252')

demog_df.info()
print("--"*30)
nis_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 21183 entries, 0 to 21182
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   CD_REFNIS            21183 non-null  int64
 1   CD_SECTOR            21183 non-null  str  
 2   TOTAL                21183 non-null  int64
 3   DT_STRT_SECTOR       21183 non-null  str  
 4   DT_STOP_SECTOR       21183 non-null  str  
 5   OPPERVLAKKTE IN HM²  21183 non-null  str  
 6   TX_DESCR_SECTOR_NL   21183 non-null  str  
 7   TX_DESCR_SECTOR_FR   21183 non-null  str  
 8   TX_DESCR_NL          21183 non-null  str  
 9   TX_DESCR_FR          21183 non-null  str  
dtypes: int64(2), str(8)
memory usage: 1.6 MB
------------------------------------------------------------
<class 'pandas.DataFrame'>
RangeIndex: 4451153 entries, 0 to 4451152
Data columns (total 21 columns):
 #   Column              Dtype  
---  ------              -----  
 0   CD_REFNIS           int64  
 1   TX_REFNIS_DES

demog_df adaptation

In [10]:
demog_df['OPPERVLAKKTE IN HM²'].head(5)

0    52,9876523968054
1    67,2440779981788
2    28,0836525518644
3    42,8105162953392
4    25,4857858896115
Name: OPPERVLAKKTE IN HM², dtype: str

In [11]:
# elements in demog_df['OPPERVLAKKTE IN HM²'] are strings and have a comma "," instead of dot "."
# I replace it and convert the datatype to float
demog_df['OPPERVLAKKTE IN HM²'] = demog_df['OPPERVLAKKTE IN HM²'].str.replace(',', '.').astype(float)
demog_df['OPPERVLAKKTE IN HM²'].head(5)

0   52.988
1   67.244
2   28.084
3   42.811
4   25.486
Name: OPPERVLAKKTE IN HM², dtype: float64

In [12]:
# add a new column to gave the info in KM² instead
demog_df['surface_KM²'] = demog_df['OPPERVLAKKTE IN HM²'] / 100
demog_df['surface_KM²'].head(5)

0   0.530
1   0.672
2   0.281
3   0.428
4   0.255
Name: surface_KM², dtype: float64

<span style="color:red">FAILED</span>: I try to merge the postal code column from nis_df to demo_df through CD_SECTOR

In [9]:
# I need population info related to postal code

# forcing conversion to str removing hidden empty spaces
nis_df['CD_SECTOR_2023'] = nis_df['CD_SECTOR_2023'].astype(str).str.strip()
demog_df['CD_SECTOR'] = demog_df['CD_SECTOR'].astype(str).str.strip()
# extract an unique mapping of postal code (CD_ZIP) for each sector (CD_SECTOR_2023) 
bridge_df = nis_df[['CD_SECTOR_2023', 'CD_ZIP']].drop_duplicates()
# renominate the column from nis_df to make it identical to the one in demog_df
bridge_df = bridge_df.rename(columns={'CD_SECTOR_2023': 'CD_SECTOR'})

In [ ]:
# joining brifge_df to demog_df
demog_df_complete = demog_df.merge(bridge_df, on='CD_SECTOR', how='inner')
print(f'Number of rows in former demog_df: {demog_df.shape[0]}')
print(f'Number of rows in new demog_df_complete: {demog_df_complete.shape[0]}')

In [ ]:
# diagnostic cell - investigate discrepancy in CD_SCETOR

# printing a sample from both starting StatBel datasets
print(f'Sample of demog_df: {demog_df['CD_SECTOR'].dropna().head(5).tolist()}')
print(f'Sample of nis_df: {nis_df['CD_SECTOR_2023'].dropna().head(5).tolist()}')

# check the length of characters
print(f'String length in demog_df: {demog_df['CD_SECTOR'].str.len().unique()}')
print(f'String length in nis_df: {nis_df['CD_SECTOR_2023'].str.len().unique()}')

<span style="color:red">My attempt failed, CD_SECTOR from demog_df and CD_SECTOR_2023 from nis_df mismatch.</span>  
  
<span style="color:green">ADOPTED APPROACH</span>: Comune (CD_REFNIS) is the only geographical reference that the twoo datasets share: so I aggregate on comune level.

In [13]:
# StatBel datasets are organized in micro-sectors
# so I need to calculate population amount and surface area in total for each comune
comune_df = demog_df.groupby('CD_REFNIS').agg(
    tot_pop_per_comune=('TOTAL', 'sum'),
    tot_area_km2=('surface_KM²', 'sum')
).reset_index()

In [14]:
# calculate and adding 'urban_density' column to comune_df
# values are 1 per comune (average representative value)

comune_df['urban_density'] = comune_df['tot_pop_per_comune'] / comune_df['tot_area_km2']

In [15]:
# isolate from nis_df correspondance between CD_ZIP and CD_REFNIS
bridge_df = nis_df[['CD_ZIP', 'CD_REFNIS']].drop_duplicates()
# add the comune_df to bridge_df to implement 'urban_density' info
df_zip_density = bridge_df.merge(comune_df[['CD_REFNIS', 'urban_density']], on='CD_REFNIS', how='inner')

# in Belgium postal_code may intersect more than one comune
# so, in order to avoid having same CD_ZIP associated to different densities, 
# final aggregation on 'CD_ZIP' calculating mean value
density_mapping = df_zip_density.groupby('CD_ZIP')['urban_density'].mean()

In [16]:
# save density_mapping as joblib file

joblib.dump(density_mapping, os.path.join(MODEL_DIR_PATH, "density_mapping.joblib"))

['../models\\density_mapping.joblib']

In [17]:
'''
properties_train
----------------
'''
print("ADDING 'urban_density' COLUMN\n")
print("properties_train")
print("----------------")

# mapping 'urban_density' to properties dataset
properties_train['urban_density'] = properties_train['postal_code'].map(density_mapping)

# checking new column state

properties_train['urban_density'].head(5)
print(f'Number on NaN values: {properties_train['urban_density'].isna().sum()}')
print(f'Proportion of NaN values: {(properties_train['urban_density'].isna().mean()*100).round(2)}%')
# check if association with 'postal_code' has geometrical sense
properties_train[['postal_code', 'urban_density']].drop_duplicates().head(10)
# extract postal codes list with NaN urban density
pc_nan_dens = properties_train[properties_train['urban_density'].isna()]['postal_code'].unique().astype(str)
print(f'Postal codes with nan as urban density:\n{"\n".join(pc_nan_dens)}')

ADDING 'urban_density' COLUMN

properties_train
----------------
Number on NaN values: 320
Proportion of NaN values: 3.23%
Postal codes with nan as urban density:
3945
3720
3980
8700
3500
3700
3740
6600
8755
3730
3668
1755
9180
9820
9120
9080
9840
9130
3840
9150
6687
9810
9160
4771
2150
9090
1570
2070
3510
8760
1540
3941
8750
3724
3511
6688


In [18]:
'''
properties_test
----------------
'''
print("ADDING 'urban_density' COLUMN\n")
print("properties_test")
print("----------------")

# mapping 'urban_density' to properties dataset
properties_test['urban_density'] = properties_test['postal_code'].map(density_mapping)

# checking new column state

properties_test['urban_density'].head(5)
print(f'Number on NaN values: {properties_test['urban_density'].isna().sum()}')
print(f'Proportion of NaN values: {(properties_test['urban_density'].isna().mean()*100).round(2)}%')
# check if association with 'postal_code' has geometrical sense
properties_test[['postal_code', 'urban_density']].drop_duplicates().head(10)
# extract postal codes list with NaN urban density
pc_nan_dens = properties_test[properties_test['urban_density'].isna()]['postal_code'].unique().astype(str)
print(f'Postal codes with nan as urban density:\n{"\n".join(pc_nan_dens)}')

ADDING 'urban_density' COLUMN

properties_test
----------------
Number on NaN values: 79
Proportion of NaN values: 3.19%
Postal codes with nan as urban density:
3500
8700
9080
2150
3511
3740
9120
9150
9820
9810
6600
3980
9185
3945
3700
3730
3840
6687
1755
8760
3721
3720
6706
1540
9130
2070
3941
3724


Adding also `'property_age'`, `'living_area_ratio'` and `'bedroom_density'` new columns.

In [19]:
'''
properties_train
----------------
'''
print("ADDING THE OTHER COLUMNS\n")
print("properties_train")
print("----------------")

'''
===========================
'property_age' column
===========================
'''
# 2026 - df['building_year']

properties_train['property_age'] = 2026 - properties_train['building_year']

'''
===========================
'living_area_ratio' column
===========================
'''
# df['living_area_m2'] / df['total_area_m2']

properties_train['living_area_ratio'] = properties_train['living_area_m2'] / properties_train['total_area_m2']

'''
===========================
'bedroom_density' column
===========================
'''
# df['bedrooms'] / df['living_area_m2']

properties_train['bedroom_density'] = properties_train['bedrooms'] / properties_train['living_area_m2']

print("✅ Columns 'property_age', 'living_area_ratio' and 'bedroom_density' successfully added to properties_train dataset.")

ADDING THE OTHER COLUMNS

properties_train
----------------
✅ Columns 'property_age', 'living_area_ratio' and 'bedroom_density' successfully added to properties_train dataset.


In [20]:
properties_train.describe(include='all')

,property_type,property_subtype,living_area_m2,bedrooms,bathrooms,postal_code,latitude,longitude,building_year,state_of_the_building,furnished,has_garage,parking_count,kitchen_equipped,has_elevator,facades,floors_total,has_garden,garden_area_m2,has_terrace,total_area_m2,epc_score,region,province,km_from_nearby_city,is_nearby_city_prestigious,floor_number,coord_swapped,urban_density,property_age,living_area_ratio,bedroom_density
count,9919,9919,9237.000,9597.000,8843.000,9919.000,9917.000,9917.000,5964.000,7660,9919.000,9919.000,3353.000,2716,9919.000,7527.000,4460.000,9919.000,2201.000,9919.000,5525.000,8361,9919,9919,9917.000,9917.000,2790.000,9919,9599.000,5964.000,5194.000,8974.000
unique,2,15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8,NaN,NaN,NaN,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11,3,11,NaN,<NA>,NaN,2,NaN,NaN,NaN,NaN
top,House,residence,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Normal,NaN,NaN,NaN,Fully equipped,NaN,NaN,NaN,NaN,NaN,NaN,NaN,C,Wallonia,Liège,NaN,<NA>,NaN,False,NaN,NaN,NaN,NaN
freq,6442,5218,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2921,NaN,NaN,NaN,1290,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1669,4745,1361,NaN,<NA>,NaN,9884,NaN,NaN,NaN,NaN
mean,NaN,NaN,172.341,3.075,1.412,5014.252,50.722,4.546,1969.568,NaN,0.036,0.450,1.940,NaN,0.190,2.957,3.320,0.537,1179.508,0.657,1481.838,NaN,NaN,NaN,35.001,0.064,3.408,NaN,1884.508,56.432,0.603,0.020
std,NaN,NaN,148.001,2.306,1.227,2682.302,0.372,0.819,47.477,NaN,0.187,0.498,32.746,NaN,0.392,0.882,2.973,0.499,5064.902,0.475,4496.044,NaN,NaN,NaN,386.809,0.244,26.820,NaN,3478.533,47.477,3.509,0.007
min,NaN,NaN,12.000,1.000,1.000,1000.000,49.507,2.509,1500.000,NaN,0.000,0.000,1.000,NaN,0.000,1.000,1.000,0.000,1.000,0.000,2.000,NaN,NaN,NaN,0.000,0.000,0.000,NaN,24.899,0.000,0.001,0.001
25%,NaN,NaN,90.000,2.000,1.000,2570.000,50.474,4.154,1950.000,NaN,0.000,0.000,1.000,NaN,0.000,2.000,2.000,0.000,117.000,0.000,270.000,NaN,NaN,NaN,3.200,0.000,1.000,NaN,253.511,19.000,0.156,0.016
50%,NaN,NaN,145.000,3.000,1.000,4960.000,50.777,4.458,1974.000,NaN,0.000,0.000,1.000,NaN,0.000,3.000,3.000,1.000,350.000,1.000,630.000,NaN,NaN,NaN,8.400,0.000,2.000,NaN,568.270,52.000,0.296,0.019
75%,NaN,NaN,205.000,4.000,2.000,7100.000,51.010,5.231,2007.000,NaN,0.000,1.000,1.000,NaN,0.000,4.000,4.000,1.000,900.000,1.000,1299.000,NaN,NaN,NaN,16.900,0.000,3.000,NaN,1728.186,76.000,0.626,0.024


In [21]:
'''
properties_test
----------------
'''
print("ADDING THE OTHER COLUMNS\n")
print("properties_test")
print("----------------")

'''
===========================
'property_age' column
===========================
'''
# 2026 - df['building_year']

properties_test['property_age'] = 2026 - properties_test['building_year']

'''
===========================
'living_area_ratio' column
===========================
'''
# df['living_area_m2'] / df['total_area_m2']

properties_test['living_area_ratio'] = properties_test['living_area_m2'] / properties_test['total_area_m2']

'''
===========================
'bedroom_density' column
===========================
'''
# df['bedrooms'] / df['living_area_m2']

properties_test['bedroom_density'] = properties_test['bedrooms'] / properties_test['living_area_m2']

print("✅ Columns 'property_age', 'living_area_ratio' and 'bedroom_density' successfully added to properties_test dataset.")

ADDING THE OTHER COLUMNS

properties_test
----------------
✅ Columns 'property_age', 'living_area_ratio' and 'bedroom_density' successfully added to properties_test dataset.


New features (added after model optimization)

In [22]:
# ===============================
#     'average_bedroom_size'
# ===============================
# Numerical
# df['living_area_m2'] / df['bedroom_count']
# Evitiamo la divisione per zero nel caso in cui bedroom_count sia 0 o NaN
properties_train['average_room_size'] = properties_train['living_area_m2'] / (properties_train['bedrooms'].replace(0, np.nan))
properties_test['average_room_size'] = properties_test['living_area_m2'] / (properties_test['bedrooms'].replace(0, np.nan))

# ===============================
#     'living_area_squared'
# ===============================
# Numerical
# 'living_area_m2' is the main feature in determining a property price
# df['living_area_m2'] ** 2
properties_train['living_area_squared'] = properties_train['living_area_m2'] ** 2
properties_test['living_area_squared'] = properties_test['living_area_m2'] ** 2

# ===============================
# 'external_area_dominance_index'
# ===============================
# Numerical (between 0 and 1)
# un buona feature complementare per gli alberi di decisione è l'indice di dominanza dello spazio 
# esterno: se il modello sa quanta parte della superficie totale non è calpestabile all'interno, 
# intercetta subito la tipologia di immobile
# 1 - df['living_area_ratio']
properties_train['external_area_dominance_index'] = 1 - properties_train['living_area_ratio']
properties_test['external_area_dominance_index'] = 1 - properties_test['living_area_ratio']

**07/06**: I need to refactor `handle_missing_values()` function for production, so I'm going to extract dataset as it is at this point and move to *preprocessing_refactoring.ipynb*.

In [23]:
TRAIN_BEFORE_HMV = "../data/raw/X_train_mid_preproc.csv"
TEST_BEFORE_HMV = "../data/raw/X_test_mid_preproc.csv"

properties_train.to_csv(TRAIN_BEFORE_HMV, index=False)
properties_test.to_csv(TEST_BEFORE_HMV, index=False)

<h3 style="color:#c8a2c8; margin-bottom: 0px; padding-bottom: 0px;">2.4 - Handling Missing Values</h3> 

In [ ]:
# calculate global medians to be saved for production preprocessing (proj 06)

# calculate medians for lat and lon grouping them by 'province'
lat_median_by_province = properties_train.groupby('province')['latitude'].median()
lon_median_by_province = properties_train.groupby('province')['longitude'].median()

# calculate national medians (for lat and lon, just in case 'province' is missing)
global_medians = {
    'latitude': df['latitude'].median(),
    'longitude': df['longitude']
}

When preparing data for a ML model, we want to preserve as many observations as possible.  
Three common strategies:
- **Drop** rows or columns (only if missing % is very high or missing is meaningful) - NOT DONE IN THIS PHASE
- **Fill with a statistic** (mean, median for numeric; mode for categorical)
- **Predict** missing values using a model (advanced) - NOT DONE IN THIS PHASE

In [19]:
# Check for nan percentages in each column of the train dataset
nan_prop = (properties_train.isnull().sum() / len(properties_train)) * 100

print("Proportion of missing values per column in properties_train before handling.\n")
print(nan_prop.sort_values(ascending=False))

# List of cols extraction
cols_50 = nan_prop[nan_prop >= 50].index.tolist()
cols_5 = nan_prop[(nan_prop <= 5) & (nan_prop > 0)].index.tolist()
cols_middle = nan_prop[(nan_prop > 5) & (nan_prop < 50)].index.tolist()

print("\n")
print(f'Columns with nan % higher than 50%:')
print(", \n".join(cols_50) if cols_50 else "None")
print("\n")
print(f'Columns with nan % lower than 5%:')
print(", \n".join(cols_5) if cols_5  else "None")
print("\n")
print(f'Columns with intermediate nan %:')
print(", \n".join(cols_middle) if cols_middle else "None")

Proportion of missing values per column in properties_train before handling.

garden_area_m2                  77.810
kitchen_equipped                72.618
floor_number                    71.872
parking_count                   66.196
floors_total                    55.036
external_area_dominance_index   47.636
living_area_ratio               47.636
total_area_m2                   44.299
building_year                   39.873
property_age                    39.873
facades                         24.115
state_of_the_building           22.774
epc_score                       15.707
bathrooms                       10.848
average_room_size                9.527
bedroom_density                  9.527
living_area_squared              6.876
living_area_m2                   6.876
bedrooms                         3.246
urban_density                    3.226
latitude                         0.020
longitude                        0.020
km_from_nearby_city              0.020
is_nearby_city_prestigiou

**NaN higher than 50%**
- `'parking_count'` &rarr; numeric, plausible to be absent if equal to NaN &rarr; **0**;
- `'kitchen_equipped'` &rarr; categorial, info's missing &rarr; **Unknown**; 
- `'floors_total'` &rarr; continous numerical variable &rarr; **median**; 
- `'floor_number'` &rarr; continous numerical variable &rarr; **median**;
- `'garden_area_m2'` &rarr; numeric, plausible to be absent if equal to NaN &rarr; **0**;

In [20]:
nan_to_0_cols_50 = ['parking_count', 'garden_area_m2']
nan_to_median_cols = ['floors_total', 'floor_number']
for col in nan_to_0_cols_50:
    properties_train[col] = properties_train[col].fillna(0)
    properties_test[col] = properties_test[col].fillna(0)
print(f'✅ NaN values in {", ".join(nan_to_0_cols_50)} columns successfully turned to 0.')
print(f'NaN values in {", ".join(nan_to_median_cols)} columns will be handled in the following cell.')

✅ NaN values in parking_count, garden_area_m2 columns successfully turned to 0.
NaN values in floors_total, floor_number columns will be handled in the following cell.


**NaN lower than 5%**: turning to median such a low percentage doesn't affect predictive power.
- `'bedrooms'` &rarr; continous numerical variable &rarr; **median**;
- `'urban_density'` &rarr; continous numerical variable &rarr; **median**;  
- `'latitude'` &rarr; handled accordingly in the cell below;  
- `'longitude'` &rarr; handled accordingly in the cell below;  
- `'nearby_city'` &rarr; categorial, info's missing &rarr; **Unknown**;  
- `'km_from_nearby_city'` &rarr; continous numerical variable &rarr; **median**;  
- `'is_nearby_city_prestigious'` &rarr; continous numerical variable &rarr; **median**;  

Since I am going to turn NaN values to median in all remaining numeric columns, they will be handled subsequently.

In [21]:
# NaN's handling of geographical coordinates

# calculating median for latitude and for longitude grouping them by province on Train
# to maintain geographical coherence
lat_median_by_province = properties_train.groupby('province')['latitude'].median()
lon_median_by_province = properties_train.groupby('province')['longitude'].median()

# nan imputing using corresponding province on Train and Test
'''
properties_train
----------------
'''
properties_train['latitude'] = properties_train['latitude'].fillna(properties_train['province'].map(lat_median_by_province))
properties_train['longitude'] = properties_train['longitude'].fillna(properties_train['province'].map(lon_median_by_province))

'''
properties_test
----------------
'''
properties_test['latitude'] = properties_test['latitude'].fillna(properties_test['province'].map(lat_median_by_province))
properties_test['longitude'] = properties_test['longitude'].fillna(properties_test['province'].map(lon_median_by_province))

print("✅ Latitude and Longitude NaN's successfully imputated to corresponding province median.")

✅ Latitude and Longitude NaN's successfully imputated to corresponding province median.


In [22]:
nan_to_median_cols += ['bedrooms', 'urban_density', 'km_from_nearby_city', 'is_nearby_city_prestigious']
print(f'NaN values to be turned to median currently in {", ".join(nan_to_median_cols)} columns.')

# for col in nan_to_median_cols_5:
#     median_value = properties_train[col].median()
#     properties_train[col] = properties_train[col].fillna(median_value)
#     properties_test[col] = properties_test[col].fillna(median_value)
# print(f'✅ NaN values in {", ".join(nan_to_median_cols_5)} columns successfully turned to median.')

NaN values to be turned to median currently in floors_total, floor_number, bedrooms, urban_density, km_from_nearby_city, is_nearby_city_prestigious columns.


**all other columns**

- turn NaN in numerical columns to **median**
- turn NaN in categorial columns to "**Unknown**"

In [23]:
nan_to_unknown_cols = properties_train.select_dtypes(exclude='number').columns

# filtering columns in middle range to select the numerical ones only
cols_middle_numerical = properties_train[cols_middle].select_dtypes(include='number').columns.tolist()
# adding them to already existing nan_to_median_cols list
nan_to_median_cols += cols_middle_numerical

# check of selected numerical columns
print(f'🔢 Numerical columns - NaN conversion to median:\n')
print("\n".join(nan_to_median_cols))
print("\n")
# check of categorial numerical columns
print(f'🔠 Categorial columns - NaN conversion to "Unknown":\n')
print("\n".join(nan_to_unknown_cols))

🔢 Numerical columns - NaN conversion to median:

floors_total
floor_number
bedrooms
urban_density
km_from_nearby_city
is_nearby_city_prestigious
living_area_m2
bathrooms
building_year
facades
total_area_m2
property_age
living_area_ratio
bedroom_density
average_room_size
living_area_squared
external_area_dominance_index


🔠 Categorial columns - NaN conversion to "Unknown":

property_type
property_subtype
state_of_the_building
kitchen_equipped
epc_score
region
province
coord_swapped


In [24]:
print("Handling Numerical columns...")
for col in nan_to_median_cols:
    median_value = properties_train[col].median()
    properties_train[col] = properties_train[col].fillna(median_value)
    properties_test[col] = properties_test[col].fillna(median_value)
print(f'✅ Nan in numerical columns successfully turned to median.')

Handling Numerical columns...
✅ Nan in numerical columns successfully turned to median.


In [25]:
print("Handling Categorial columns...")
properties_train[nan_to_unknown_cols] = properties_train[nan_to_unknown_cols].fillna('Unknown')
properties_test[nan_to_unknown_cols] = properties_test[nan_to_unknown_cols].fillna('Unknown')
print(f'✅ Nan in categorial columns successfully turned to "Unknown".')

Handling Categorial columns...
✅ Nan in categorial columns successfully turned to "Unknown".


<h3 style="color:#c8a2c8; margin-bottom: 0px; padding-bottom: 0px;">2.5 - Encoding Categorical Features</h3>  
<h4 style="color:#c8a2c8; margin-top: 0px; margin-bottom: 0px; padding-top: 0px; padding-bottom: 0px;">Encoding on both properties_train and properties_test.</h4>
<h4 style="color:#c8a2c8; margin-top: 0px; padding-top: 0px;">
    <code>.fit_transform()</code> only on properties_train (X_train_raw)<br>
    <code>.transform()</code> only on properties_test (X_test_raw)
</h4>

Using <span style="color:#8064A2; font-weight:bold">Label / Ordinal Encoding</span> for variables that categorize data according to a specific sequence (**ordinal variables**: the numerical order reflects reality - like `'epc_score'`) and <span style="color:#2E75B5; font-weight:bold">One-Hot Encoding</span> for nominal variables that don't (**nominal variables** - like `'property_subtype'`).

Categorical columns in properties df are:  
- `'property_type'` &rarr; `House` (<span style="color:orange">change to 0</span>) and `Apartment` (<span style="color:orange">change to 1</span>); 
- `'property_subtype'` (<span style="color:#2E75B5; font-weight:bold">One-Hot Encoding</span>) &rarr; `residence`, `apartment`, `duplex`, `mixed-building`, `bungalow`, `villa`, `penthouse`, `cottage`, `chalet`, `studio`, `ground-floor`, `triplex`, `master-house`, `mansion` and `loft`;
- `'state_of_the building'` (<span style="color:#8064A2; font-weight:bold">Label / Ordinal Encoding</span>) &rarr; <span style="color:orange">make a MAPPING DICTIONARY (unify `Fully renovated` and `Excellent`)</span>: `New`, `Under construction`, `Fully renovated`, `Normal`, `To renovate`, `To restore` and `To demolish`;
- `'furnished'` &rarr; <span style="color:green">DONE</span>: already binary, dtype int64;
- `'has_garage'` &rarr; <span style="color:green">DONE</span>: already binary, dtype int64;
- `'kitchen_equipped'` (<span style="color:#8064A2; font-weight:bold">Label / Ordinal Encoding</span>) &rarr; <span style="color:orange">make a MAPPING DICTIONARY</span>: `Super equipped`, `Fully equipped`, `Partially equipped` and `Not equipped`;
- `'has_elevator'` &rarr; <span style="color:green">DONE</span>: already binary, dtype int64;
- `'has_garden'` &rarr; <span style="color:green">DONE</span>: already binary, dtype int64;
- `'has_terrace'`  &rarr; <span style="color:green">DONE</span>: already binary, dtype int64;
- `'epc_score'` (<span style="color:#8064A2; font-weight:bold">Label / Ordinal Encoding</span>) &rarr; <span style="color:orange">add intermediate levels `A-`, `B-`, `C-`, `D-`, `E-`, `C+` and `D+` and make a MAPPING DICTIONARY</span>: `A++`, `A+`, `A`, `B+`, `B`, `C`, `D`, `E+`, `E`, `F`, `G`;
- `'region'` (<span style="color:#2E75B5; font-weight:bold">One-Hot Encoding</span>) &rarr; `Wallonia`, `Brussels` and `Flanders`;
- `'province'` (<span style="color:#2E75B5; font-weight:bold">One-Hot Encoding</span>) &rarr; `Liège`, `Brussels Capital`, `Region`, `Hainaut`, `Limburg`, `Luxembourg`, `Namur`, `East Flanders`, `Antwerp`, `West Flanders`, `Walloon Brabant` and `Flemish Brabant`;
- `'is_nearby_city_prestigious'` &rarr; <span style="color:yellow">CHANGE DTYPE TO INT64</span>: already binary, but change from float64 to int64;
- `'urban_density'` &rarr; <span style="color:orange">ORGANIZE IN 3 CATEGORIES</span>: continuous numerical variable, to be splitted in 3 categories with Quantile Binning.
 

<span style="color:#8064A2; font-weight:bold">Ordinal Encoding</span> is exclusively for the input features X (the feature columns, such as epc_score or state_of_the_building),

<div style="margin-top: 0px; margin-bottom: 0px; padding-top: 0px; padding-bottom: 0px;">
    <strong>property_type</strong><br>
    Currently contains `House` and `Apartment`;<br>
    must be replaced respectively with 0 and 1.
</div>

In [26]:
# replacing nominals with numbers
type_mapping = {
    'House': 0,
    'Apartment': 1
}
properties_train['property_type'] = properties_train['property_type'].str.capitalize().replace(type_mapping).astype("Int64")
properties_test['property_type'] = properties_test['property_type'].str.capitalize().replace(type_mapping).astype("Int64")

print("✅ 'property_type' column successfully numerized in both datasets.")

✅ 'property_type' column successfully numerized in both datasets.


<div style="margin-top: 0px; margin-bottom: 0px; padding-top: 0px; padding-bottom: 0px;">
    <strong>urban_density</strong><br>
    Currently contains numerical values;<br>
    must be displaied in order from best to worse in 3 categories.
</div>

<i><span style="color:orange">Note for the 03/07:</span> make <code>'urban_density_encoded'</code> be displaied as <code>'urban_density'</code>.</i>

In [27]:
# check numerical data distribution for optimal subdivision in 3 nominal categories
# quantile banning

# directly generating 0, 1 and 2 on Train
properties_train['urban_density_encoded'], bins_density = pd.qcut(
    properties_train['urban_density'],
    q=3,
    labels=False,
    retbins=True
)

# apply same bins to Test
properties_test['urban_density_encoded'] = pd.cut(
    properties_test['urban_density'],
    bins=bins_density,
    labels=False,
    include_lowest=True
)

print("✅ urban_density column successfully grouped in 3 categories in both datasets.")
print("⛔ No need for it to be included in the OrdinalEncoder.")

✅ urban_density column successfully grouped in 3 categories in both datasets.
⛔ No need for it to be included in the OrdinalEncoder.


<h4 style="color:#2E75B5; font-weight:bold; margin-bottom: 0px; padding-bottom: 0px">One-Hot Encoding</h4>  

On columns:  
- `property_subtype`;  
- `region`;  
- `province`.

ML pipeline approach: <span style="color:#458b74">sklearn.OneHotEncoder</span>

In [28]:
# categorial features
categ_cols = ['property_subtype', 'region', 'province']

# Initialization with unforseen events managing
encoder = OneHotEncoder(
    handle_unknown='ignore', 
    sparse_output=False)
    # sparse_output -> to make Pandas return a clean dataframe keeping columns original names
    # handle_unknown -> to gracefully handle unknown values that may be present in test set
encoder.set_output(transform='pandas') # to be paired with sparse_output=False

'''
properties_train
----------------
'''
# Training AND Transformation on TRAIN
X_train_categ_encoded = encoder.fit_transform(properties_train[categ_cols])

properties_train = properties_train.drop(columns=categ_cols)
properties_train = pd.concat([properties_train, X_train_categ_encoded], axis=1)

print("✅ properties_train dataset successfully trained and transformed.")

'''
properties_test
----------------
'''
# only Transformation on TEST
X_test_categ_encoded = encoder.transform(properties_test[categ_cols])

properties_test = properties_test.drop(columns=categ_cols)
properties_test = pd.concat([properties_test, X_test_categ_encoded], axis=1)

print("✅ properties_test dataset successfully transformed.")


✅ properties_train dataset successfully trained and transformed.
✅ properties_test dataset successfully transformed.


<h4 style="color:#8064A2; font-weight:bold; margin-bottom: 0px; padding-bottom: 0px">Ordinal Encoding</h4>

On columns:  
- `state_of_the_building`;  
- `kitchen_equipped`;  
- `epc_score`.

<span style="font-weight:bold; margin-bottom: 0px; padding-bottom: 0px">state_of_the_building</span>  
<span style="margin-top: 0px; margin-bottom: 0px; padding-top: 0px; padding-bottom: 0px;">contains in order from best to worse <code>`New`, `Under construction`, `Fully renovated`, `Normal`, `To renovate`, `To restore`, `To demolish`</code>.</span>  
Unify `Fully renovated` and `Excellent` under `Fully renovated`.

In [29]:
# replacing Excellent category with Fully renovated
redundancies_mapping = {
    'Excellent': 'Fully renovated',
}
properties_train['state_of_the_building'] = properties_train['state_of_the_building'].replace(redundancies_mapping)
properties_test['state_of_the_building'] = properties_test['state_of_the_building'].replace(redundancies_mapping)

# logical order definition
order_state_of_the_building = ['To demolish', 'To restore', 'To renovate', 'Normal', 'Fully renovated', 'under construction', 'New']
# Under construction is placed at second place because 
# is seen to be as the same market value level of a new property,
# but is limited due to not immediate availability.

print("🚦 'state_of_the_building' column is ready for OrdinalEncoder.")

🚦 'state_of_the_building' column is ready for OrdinalEncoder.


<span style="font-weight:bold; margin-bottom: 0px; padding-bottom: 0px">kitchen_equipped</span>  
<span style="margin-top: 0px; margin-bottom: 0px; padding-top: 0px; padding-bottom: 0px;">contains in order from best to worse <code>`Super equipped`, `Fully equipped`, `Partially equipped`, `Not equipped`</code>.</span>

In [30]:
# logical order definition
order_kitchen_equipped = ['Not equipped', 'Partially equipped', 'Fully equipped', 'Super equipped']

print("🚦 'kitchen_equipped' column is ready for OrdinalEncoder.")

🚦 'kitchen_equipped' column is ready for OrdinalEncoder.


<span style="font-weight:bold; margin-bottom: 0px; padding-bottom: 0px">epc_score</span>  
<span style="margin-top: 0px; margin-bottom: 0px; padding-top: 0px; padding-bottom: 0px;">contains in order from best to worse <code>`A++`, `A+`, `A`, `B+`, `B`, `C`, `D`, `E+`, `E`, `F`, `G`.</code></span>  
Add in the right order <code>`A-`, `B-`, `C-`, `D-`, `E-`, `C+`, `D+`</code>.

In [31]:
# logical order definition with addictional categories
order_epc_score = ['G', 'F', 'E-', 'E', 'E+', 'D-', 'D', 'D+', 'C-', 'C', 'C+', 'B-', 'B', 'B+', 'A-', 'A', 'A+', 'A++']

print("🚦 'epc_score' column is ready for OrdinalEncoder.")

🚦 'epc_score' column is ready for OrdinalEncoder.


In [32]:
# INITIALIZATION

# cols to elaborate
cols_to_elaborate = ['state_of_the_building', 'kitchen_equipped', 'epc_score']

# lists with categories in right order for each column have been declared above
# order_<column_name>

# Encoder Configuration
# lists must be associated in exact same order as the columns
encoder = OrdinalEncoder(
    categories=[order_state_of_the_building, order_kitchen_equipped, order_epc_score],
    handle_unknown='use_encoded_value', 
    unknown_value=-1
)
    # handle_unknown='use_encoded_value' prevents crashing in case of unknown strings in train

print("✅ OrdinalEncoder successfully initialized.")

✅ OrdinalEncoder successfully initialized.


In [33]:
# EXECUTION

print("Execution on properties_train")
print("-----------------------------")
'''
properties_train
----------------
'''
# Training and Transformation
properties_train[cols_to_elaborate] = encoder.fit_transform(properties_train[cols_to_elaborate])
print(f'✅ properties_train dataset successfully trained.')
print(f'✅ Columns {", ".join(cols_to_elaborate)} in properties_train dataset successfully encoded.')

print("\nExecution on properties_test")
print("----------------------------")
'''
properties_test
----------------
'''
# Transformation
properties_test[cols_to_elaborate] = encoder.transform(properties_test[cols_to_elaborate])
print(f'✅ Columns {", ".join(cols_to_elaborate)} in properties_test dataset successfully encoded.')

Execution on properties_train
-----------------------------
✅ properties_train dataset successfully trained.
✅ Columns state_of_the_building, kitchen_equipped, epc_score in properties_train dataset successfully encoded.

Execution on properties_test
----------------------------
✅ Columns state_of_the_building, kitchen_equipped, epc_score in properties_test dataset successfully encoded.


<h3 style="color:#c8a2c8; margin-bottom: 0px; padding-bottom: 0px;">2.6 - Feature Scaling</h3>  
<h4 style="color:#c8a2c8; margin-top: 0px; margin-bottom: 0px; padding-top: 0px; padding-bottom: 0px;">Normalisation & Standardisation</h4>

I check all numerical columns and divide them according to their values.

In [34]:
all_numerical_cols = properties_train.select_dtypes(exclude=['object', 'category']).columns

# check of all numerical columns in properties_train
cols_0_1 = []
cols_0_1_2 = []
other_num_cols = []
print(f'🔢 Numerical columns in properties_train:\n')
for col in all_numerical_cols:
    if properties_train[col].isin([0,1]).all():
        cols_0_1.append(col)
    elif properties_train[col].isin([0,1,2]).all():
        cols_0_1_2.append(col)
    else:
        other_num_cols.append(col)
print(f'Columns with 0 and 1 only:\n')
print(", \n".join(cols_0_1))
print(f'\nColumns with 0, 1 and 2 only:\n')
print(", \n".join(cols_0_1_2))
print(f'\nAll other numerical columns:\n')
print(", \n".join(other_num_cols))

🔢 Numerical columns in properties_train:

Columns with 0 and 1 only:

property_type, 
furnished, 
has_garage, 
has_elevator, 
has_garden, 
has_terrace, 
is_nearby_city_prestigious, 
coord_swapped, 
property_subtype_apartment, 
property_subtype_bungalow, 
property_subtype_chalet, 
property_subtype_cottage, 
property_subtype_duplex, 
property_subtype_ground-floor, 
property_subtype_loft, 
property_subtype_mansion, 
property_subtype_master-house, 
property_subtype_mixed-building, 
property_subtype_penthouse, 
property_subtype_residence, 
property_subtype_studio, 
property_subtype_triplex, 
property_subtype_villa, 
region_Brussels, 
region_Flanders, 
region_Wallonia, 
province_Antwerp, 
province_Brussels Capital Region, 
province_East Flanders, 
province_Flemish Brabant, 
province_Hainaut, 
province_Limburg, 
province_Liège, 
province_Luxembourg, 
province_Namur, 
province_Walloon Brabant, 
province_West Flanders

Columns with 0, 1 and 2 only:

urban_density_encoded

All other numerical co

In this case, Standardisation (<code>StandardScaler</code>, 0 as mean, 1 as variance) is the right choice over Normalisation (<code>MinMaxScaler</code>, 0 as lowest, 1 as highest), because properties dataset inherently has outliers that using MinMaxScaler would compress the vast majority of "normal" data into a tiny interval close to zero. This would kill the predicting power of the resulting model.

Binary columns (columns containing only 0 and 1), <code>'urban_density'</code> (contains only 0, 1 and 2) and columns just transformed with OrdinalEncoder (<code>'state_of_the_building'</code>, <code>'kitchen_equipped'</code> and <code>'epc_score'</code>) which maintain a significant, distinct scale, and <code>'postal_code'</code>, which is a geographic identifier, must be excluded by the Standardisation phase.

In [35]:
cols_to_exclude = ['state_of_the_building', 'kitchen_equipped', 'epc_score', 'postal_code']

# python list to NumPy array
idx_num = pd.Index(other_num_cols)
# generation of array of boolean values
masq = idx_num.isin(cols_to_exclude)
# boolean indexing (extraction with pandas of only True elements and their groupin within a list)
cols_to_scale = idx_num[~masq].tolist()

# Initialisation
scaler = StandardScaler()

# Execution
# Train
print("EXECUTION ON properties_train...")
properties_train[cols_to_scale] = scaler.fit_transform(properties_train[cols_to_scale])
print("✅ Selected columns successfully scaled with StandardScaler in properties_train dataset.")

# Test
print("EXECUTION ON properties_test...")
properties_test[cols_to_scale] = scaler.transform(properties_test[cols_to_scale])
print("✅ Selected columns successfully scaled with StandardScaler in properties_test dataset.")

EXECUTION ON properties_train...
✅ Selected columns successfully scaled with StandardScaler in properties_train dataset.
EXECUTION ON properties_test...
✅ Selected columns successfully scaled with StandardScaler in properties_test dataset.


<h3 style="color:#c8a2c8; margin-bottom: 0px; padding-bottom: 0px;">2.7 - To the <code>sklearn</code> Pipeline</h3>

Check if there are still any strings in the dataset

In [36]:
cols_with_str = properties_train.select_dtypes(include=['object', 'category', 'string']).columns.tolist()

if not cols_with_str:
    print("✅ There are no columns containing strings left in the train dataset, you can proceed to the Baseline Model phase.")
else:
    print(f"⛔ There are still columns containing strings in the train dataset:\n{"\n".join(cols_with_str)}.")

✅ There are no columns containing strings left in the train dataset, you can proceed to the Baseline Model phase.


In [37]:
PROP_TRAIN_PREPROC_CSV_PATH = "../data/cleaned/properties_train_preprocessed.csv"
PROP_TEST_PREPROC_CSV_PATH = "../data/cleaned/properties_test_preprocessed.csv"

properties_train.to_csv(PROP_TRAIN_PREPROC_CSV_PATH, index=False)
properties_test.to_csv(PROP_TEST_PREPROC_CSV_PATH, index=False)

print(f'✅ Train dataset successfully saved as "properties_train_preprocessed.csv" in {PROP_TRAIN_PREPROC_CSV_PATH}')
print(f'✅ Test dataset successfully saved as "properties_test_preprocessed.csv" in {PROP_TEST_PREPROC_CSV_PATH}')

✅ Train dataset successfully saved as "properties_train_preprocessed.csv" in ../data/cleaned/properties_train_preprocessed.csv
✅ Test dataset successfully saved as "properties_test_preprocessed.csv" in ../data/cleaned/properties_test_preprocessed.csv


<i><span style="color:orange">Note for the 02/07:</span> Think about how to handle outliers and update preprocessing.</i>